# Classification du jeu Imagenet-1K avec Alexnet

- Creation : *18/02/2025*

Constat de la performance de classification du modèle.
Utilisation de la définition du modèle et des poids disponibles dans PyTorch.

- [ ] Essayer AlexNet_Weights.IMAGENET1K_V1.transforms, ensemble des transformations prèdéfinie dédié à AlexNet entrainé sur Imagenet-1K.
- [ ] Evaluations de l'erreur de validation sur les jeux de données Imagenet-1K avec 1K images (imagenet-sample-images-master) et 50K images(imagenet_val_images) disponibles localement. 


# Module

In [1]:
import os
from collections import Counter, defaultdict

import torch
import torchvision
from torchvision import transforms as T
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from datasets import DATASET_1, DATASET_2, CustomImageDataset, get_label_data_from_filename
from utils.alexnet_for_deconv import alexnet_for_deconv
from utils.utils_images import display_image_tensor as display_image_tensor_
from imagenet_labels import imagenet1K_labels_to_names

In [2]:
# Spécifiquement pour un carnet de type Jupyter
def display_image_tensor(img_tensor, verbose=True):
    if display:
        display_image_tensor_(img_tensor, verbose=verbose, fn_display=display)

In [3]:
test_debug = True
do_validation = False
TORCHVISION_MODELS_WEIGHTS = torchvision.models.AlexNet_Weights

# Device

In [4]:
device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"{device} device available")

mps device available


# Chargement des données

Création du dataset sur les images

In [5]:
DATASET = DATASET_1

In [6]:
"""
from datasets import imagenet_mean, imagenet_std

geo_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
])

transforms = T.Compose([
    geo_transforms,
    T.Lambda(lambda t: t/255.), # because read_image -> [0..255]
    T.Normalize(mean=imagenet_mean, std=imagenet_std),
])
""";

transforms = TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1.transforms()

# Display the transforms
# Mais attention, contrairement à ce qui serait affiché, le resize est exécuté avant le crop
print("Transforms", transforms)

get_label_data = lambda f: get_label_data_from_filename(f, DATASET["path"])
dataset_path = DATASET["mounted_path"] if os.path.exists(DATASET["mounted_path"]) else DATASET["path"]
print("Dataset (name, path)", DATASET["name"], dataset_path)

dataset = CustomImageDataset(
    dataset_path,
    transform=transforms,
    extension="JPEG",
    dataset_mode=True,
    only_label_idx=False,
    get_label_data=get_label_data,
    )

Transforms ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)
Dataset (name, path) imagenet_val_images (50K images) /Users/me/Documents/Work/Dev/_data/imagenet_val_images


In [7]:
dataset[1]

(tensor([[[0.8789, 0.9132, 0.9474,  ..., 1.4954, 1.4612, 1.4612],
          [0.8961, 0.8789, 0.9303,  ..., 1.4612, 1.4612, 1.4954],
          [0.8961, 0.8789, 0.8618,  ..., 1.4612, 1.4954, 1.4783],
          ...,
          [1.4269, 1.3927, 1.3070,  ..., 1.8037, 1.7694, 1.7523],
          [1.4440, 1.5297, 1.4612,  ..., 1.8037, 1.8208, 1.7865],
          [1.4783, 1.5125, 1.4954,  ..., 1.7865, 1.7694, 1.7865]],
 
         [[1.2206, 1.1856, 1.1681,  ..., 1.7458, 1.7808, 1.8333],
          [1.2031, 1.1506, 1.1681,  ..., 1.7808, 1.8158, 1.7983],
          [1.2031, 1.1681, 1.2206,  ..., 1.8158, 1.7983, 1.7808],
          ...,
          [1.6933, 1.6758, 1.6408,  ..., 2.0434, 2.0784, 2.1134],
          [1.6933, 1.8158, 1.7633,  ..., 2.0784, 2.0959, 2.1310],
          [1.6758, 1.7108, 1.7108,  ..., 2.0959, 2.1134, 2.1310]],
 
         [[1.4897, 1.5071, 1.5420,  ..., 1.8731, 1.8905, 1.9254],
          [1.5071, 1.4897, 1.4897,  ..., 1.8905, 1.8905, 1.8731],
          [1.5071, 1.5071, 1.5420,  ...,

In [8]:
batch_size = 32
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

Test dataloader

In [9]:
if test_debug:
    i_element = 1
    batch = next(iter(dataloader))
    print("Dimension batch:", len(batch), end="\n\n")
    
    print("Première partie du Batch :", batch[0].size())
    print("Deuxième partie du Batch :", batch[1].size())
    print("Troisième partie du Batch :", len(batch[2]))
    print("Quatrième partie du Batch :", batch[3].size(), end="\n\n")
    
    print("Image :", batch[0][i_element].size())
    print("Idx etiquette :", batch[1][i_element].item())
    print("Code etiquette :", batch[2][i_element])
    print("Id file :", batch[3][i_element].item())

Dimension batch: 4

Première partie du Batch : torch.Size([32, 3, 224, 224])
Deuxième partie du Batch : torch.Size([32])
Troisième partie du Batch : 32
Quatrième partie du Batch : torch.Size([32])

Image : torch.Size([3, 224, 224])
Idx etiquette : 682
Code etiquette : n03837869
Id file : 22525


### Validation de la concordance des informations de classifications

In [10]:
if do_validation:
    from ILSVRC2012_synsets import ILSVRC2012_synsets_idx_to_codes

    def read_imagenet_classes_ground_truth(file_path = "ILSVRC2012_validation_ground_truth.txt"):
        with open(file_path, 'r') as f:
            class_ids = [int(line.strip()) for line in f if line.strip().isdigit()]
        return class_ids

    # Exemple d'utilisation
    imagenet_classes_ground_truth = read_imagenet_classes_ground_truth()
    print("Nombre de lignes :", len(imagenet_classes_ground_truth))

    for filename, ground_truth_idx in zip(dataset.files, imagenet_classes_ground_truth):
        label_code, label_idx, _ = get_label_data_from_filename(filename, DATASET["path"])
        ground_truth_code = ILSVRC2012_synsets_idx_to_codes[ground_truth_idx]
        assert ground_truth_code == label_code, f"Discordance: {(ground_truth_code, ground_truth_idx)} != {(label_code, label_idx)} pour {filename}"
    print("Tous les labels concordent.")

# Chargement du modèle

In [11]:
#model_alexnet_deconv = alexnet_for_deconv(weights='IMAGENET1K_V1')
#model_alexnet_deconv.eval()
#model_alexnet_deconv.to(device)
model_name = "alexnet"
model_convnet = torch.hub.load('pytorch/vision', model_name, weights=TORCHVISION_MODELS_WEIGHTS.IMAGENET1K_V1)

Using cache found in /Users/me/.cache/torch/hub/pytorch_vision_main


In [ ]:
if test_debug:
    model_convnet.eval()
    model_convnet.to(device)

    i_image = 10
    batch_input = batch[0][i_image].unsqueeze(dim=0)
    print("Batch input :", batch_input.size())
    output = model_convnet.forward(batch_input.to(device))

    print(output.size())
    probabilities = torch.nn.functional.softmax(output, dim=1)
    predicted = probabilities.argmax(dim=1).to("cpu")
    print("predicted idx :", predicted)
    expected = batch[1][i_image]

    print("expected :", expected.tolist())

    #print(probabilities.argmax(dim=1).to("cpu") == batch[1][0])
    #print((probabilities.argmax(dim=1).to("cpu") == batch[1][0]).sum())

In [ ]:
if test_debug:
    model_convnet.eval()
    model_convnet.to(device)
    predicted_labels_prob = model_convnet(batch[0].to(device)).to("cpu")
    print(predicted_labels_prob.size())
    print(predicted_labels_prob.argmax(dim=1))
    print(torch.nn.functional.softmax(predicted_labels_prob, dim=1).argmax(dim=1))

# Classification

Il faut espérer que l'indexation de la sortie du classifier correspond à celle des classes de l'ImageNet-1K récupérée dans la liste 1K.

### Version softmax + topk

In [12]:
all_predictions_batch = []
all_expecteds_batch = []
model_convnet.eval()
model_convnet.to(device)
for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    output = model_convnet.forward(batch[0].to(device)) # To carefully transfer to CPU before append in list
    output = output.to("cpu")
    probabilities = torch.nn.functional.softmax(output, dim=1)
    top5_prob, top5_catid = torch.topk(probabilities, 5)
    all_predictions_batch.append((top5_prob.tolist(), top5_catid.tolist()))
    all_expecteds_batch.append(batch[1].tolist())

  0%|          | 0/1563 [00:00<?, ?it/s]

In [13]:
from collections import Counter
distribution_computed_top1 = Counter()
distribution_computed_top5 = Counter()
distribution_expected = Counter()

right_top1 = 0
right_top5 = 0

for (predictions_batch_output, predictions_batch_catid), expected_batch in zip(all_predictions_batch, all_expecteds_batch):
    predictions_batch_catid = torch.tensor(predictions_batch_catid)
    expected_batch = torch.tensor(expected_batch).reshape(-1, 1)

    expected_in_top1 = (predictions_batch_catid[:, 0] - expected_batch) == 0
    expected_in_top5 = (predictions_batch_catid - expected_batch) == 0

    right_top1 += expected_in_top1.sum().item()
    right_top5 += expected_in_top5.sum().item()

    distribution_computed_top1.update(predictions_batch_catid[:, 0].ravel().tolist())
    distribution_computed_top5.update(predictions_batch_catid.ravel().tolist())
    distribution_expected.update(expected_batch.ravel().tolist())

Justesse top1: 29769 (59.54%)
Justesse top5: 39554 (79.11%)


In [15]:
print(f"Justesse Top-1: {right_top1} ({right_top1 / len(dataset) * 100:.2f}%) | Error: {100. - right_top1 / len(dataset) * 100:.2f}%")
print(f"Justesse Top-5: {right_top5} ({right_top5 / len(dataset) * 100:.2f}%) | Error: {100. - right_top5 / len(dataset) * 100:.2f}%")

Justesse Top-1: 29769 (59.54%) | Error: 40.46%
Justesse Top-5: 39554 (79.11%) | Error: 20.89%


In [ ]:
print(distribution_computed_top1.total())
print(distribution_computed_top1.most_common(10))
print(distribution_computed_top5.total())
print(distribution_computed_top5.most_common(10))
list(distribution_expected.items())[:10]

Calcul des ratios par classe

Voir pour la matrice de confusion ?

In [ ]:
def counter_to_tensor(counter, size=10):
    a = [0] * size
    for k, v in counter.items():
        a[k] = v
    return torch.tensor(a)

distribution_expected_t = counter_to_tensor(distribution_expected, len(imagenet1K_labels_to_names))
distribution_computed_t = counter_to_tensor(distribution_computed_top1, len(imagenet1K_labels_to_names))

print((distribution_computed_t - distribution_expected_t).abs().sum())

print("Equal distrib count", (distribution_computed_t == distribution_expected_t).sum().item())
print("Equal distrib ratio", (distribution_computed_t == distribution_expected_t).sum() / len(dataset))


#print("Equal distrib ratio by class", (distribution_computed_t - distribution_expected_t).abs() / torch.where(distribution_expected_t == 0, torch.tensor(1), distribution_expected_t))

#print(distribution_computed_t)
#print(distribution_expected_t)

### Version sans softmax + topk

In [ ]:
all_predictions_batch = []
all_expecteds_batch = []
model_convnet.eval()
model_convnet.to(device)
for i, batch in tqdm(enumerate(dataloader), total=len(dataloader)):
    output = model_convnet.forward(batch[0].to(device)) # To carefully transfer to CPU before append in list
    output = output.to("cpu")
    top5_output, top5_catid = torch.topk(output, 5)
    all_predictions_batch.append((top5_output.tolist(), top5_catid.tolist()))
    all_expecteds_batch.append(batch[1].tolist())

In [ ]:
from collections import Counter
distribution_computed_top1 = Counter()
distribution_computed_top5 = Counter()
distribution_expected = Counter()

right_top1 = 0
right_top5 = 0

for (predictions_batch_output, predictions_batch_catid), expected_batch in zip(all_predictions_batch, all_expecteds_batch):
    predictions_batch_catid = torch.tensor(predictions_batch_catid)
    expected_batch = torch.tensor(expected_batch).reshape(-1, 1)

    expected_in_top1 = (predictions_batch_catid[:, 0] - expected_batch) == 0
    expected_in_top5 = (predictions_batch_catid - expected_batch) == 0

    right_top1 += expected_in_top1.sum().item()
    right_top5 += expected_in_top5.sum().item()

    distribution_computed_top1.update(predictions_batch_catid[:, 0].ravel().tolist())
    distribution_computed_top5.update(predictions_batch_catid.ravel().tolist())
    distribution_expected.update(expected_batch.ravel().tolist())


print(f"Justesse top1: {right_top1} ({right_top1 / len(dataset) * 100:.2f}%)")
print(f"Justesse top5: {right_top5} ({right_top5 / len(dataset) * 100:.2f}%)")